# MCM LTER Sample Log Cleanup

This script was created by Jared Collins in 2026 to be used by the MCM LTER stream team as an easy way to organize data from the pH record as recorded in the sample list each season. Currently, pH is measured using a YSI ProDSS Water Quality meter on stream water _in situ_. Originally, however, pH was measured during the chemistry measurements after being collected and sent back to lab, so there could be some discrepancy in measurements made over time.

## Importing Packages + Setting Directory

It is very important that the computer being used for processing contains the proper packages, paths, and names for the data files that will be used. The hyporheic data is a little bit more difficult to work with than data processed in Aquarius- unlike data from Aquarius, it is not automatically converted into a dataframe from the .dat file.

**To use this script, please copy the .xlxs file you would like to convert to your downloads folder.**

Note: The datafiles from the CR1000X all have the same name with no date differentiation, so this can only be done using one data file per stream at a time.

In [8]:
import pandas as pd
import glob
from pathlib import Path
import re
import numpy as np

In [3]:
home = Path.home()
downloads_path = home / 'Downloads'
data_path = downloads_path / 'Sample Logs'
files = glob.glob(str(data_path / '*.xlsx'))
dfs = []
for file in files:
    df = pd.read_excel(file)
    df['source_file'] = Path(file).name
    dfs.append(df)
combined = pd.concat(dfs, ignore_index=True)

print(combined.head())

C:\Users\jared\miniconda3\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
C:\Users\jared\miniconda3\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
C:\Users\jared\miniconda3\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


        Date Collected  Time        Date Filtered              Sample Name  \
0  2015-11-22 00:00:00  1541  2015-11-23 00:00:00          Santa Fe Stream   
1  2015-11-22 00:00:00  1957  2015-11-23 00:00:00            Wharton Creek   
2  2015-11-24 00:00:00  1533  2015-11-24 00:00:00  C1, Commonwealth Stream   
3  2015-11-25 00:00:00  1224  2015-11-25 00:00:00                M1, Adams   
4  2015-11-26 00:00:00  1725  2015-11-26 00:00:00       H1, Anderson Creek   

     pH Conductivity                                           Comments  \
0  8.38          NaN  Filtered one day late. Acidified two days late...   
1  8.08          NaN     Filtered one day late. Acidified two days late   
2  7.49        SC 96                            Alkalinity bottle froze   
3   NaN          NaN                                                NaN   
4  6.13      AC 59.9  AC = Actual Conductivity - water too cold for ...   

                                          Unnamed: 7  \
0  Several bottles found

In [4]:
column_keywords = {
    "conductivity": [r"conductivity", r"\bcond\b", r"\bspc\b", r"\bsc\b"],
    "temperature": [r"\btemp\b", r"temperature", r"\bwt\b"],
    "dissolved_oxygen": [r"\bdo\b", r"dissolved oxygen"],
    "sample_name": [r"\bstream\b", r"\bsample name\b"],
    "sample_time": [r"\btime\b", r"\bsample time\b"],
    "ph": ["ph", "ph (field)", "ph (lab)"]
}

def standardize_column(col_name):
    col_lower = str(col_name).lower()

    for standard_name, patterns in column_keywords.items():
        for pattern in patterns:
            if re.search(pattern, col_lower):
                return standard_name

    return col_name

combined.columns = [
    standardize_column(col)
    for col in combined.columns
]
combined.columns
print(pd.Series(combined.columns).value_counts())

conductivity        4
ph                  3
sample_time         2
sample_name         2
temperature         2
Date Filtered       1
Comments            1
Date Collected      1
Unnamed: 7          1
source_file         1
Sample #            1
Unnamed: 9          1
Unnamed: 11         1
dissolved_oxygen    1
Turb (field)        1
Unnamed: 13         1
Name: count, dtype: int64


In [5]:
def merge_duplicates(df):
    out = pd.DataFrame(index=df.index)
    for col in df.columns.unique():
        block = df.loc[:, df.columns == col]
        if block.shape[1] == 1:
            out[col] = block.iloc[:, 0]
        else:
            out[col] = block.bfill(axis=1).iloc[:, 0]
    return out

df = merge_duplicates(combined)
df["Date Collected"] = pd.to_datetime(df["Date Collected"], errors="coerce")

df = df.dropna(subset=["Date Collected"])
df

C:\Users\jared\AppData\Local\Temp\ipykernel_39384\2347639822.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out[col] = block.bfill(axis=1).iloc[:, 0]
C:\Users\jared\AppData\Local\Temp\ipykernel_39384\2347639822.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out[col] = block.bfill(axis=1).iloc[:, 0]
C:\Users\jared\AppData\Local\Temp\ipykernel_39384\2347639822.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in t

,Date Collected,sample_time,Date Filtered,sample_name,ph,conductivity,Comments,Unnamed: 7,source_file,Sample #,temperature,Unnamed: 9,Unnamed: 11,dissolved_oxygen,Turb (field),Unnamed: 13
0,2015-11-22,1541.0,2015-11-23 00:00:00,Santa Fe Stream,8.38,NaN,Filtered one day late. Acidified two days late...,Several bottles found frozen in refrigerator -...,Stream Team Sample List 1516.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-11-22,1957.0,2015-11-23 00:00:00,Wharton Creek,8.08,NaN,Filtered one day late. Acidified two days late,NaN,Stream Team Sample List 1516.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-11-24,1533.0,2015-11-24 00:00:00,"C1, Commonwealth Stream",7.49,SC 96,Alkalinity bottle froze,NaN,Stream Team Sample List 1516.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-11-25,1224.0,2015-11-25 00:00:00,"M1, Adams",NaN,NaN,NaN,NaN,Stream Team Sample List 1516.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-11-26,1725.0,2015-11-26 00:00:00,"H1, Anderson Creek",6.13,AC 59.9,AC = Actual Conductivity - water too cold for ...,NaN,Stream Team Sample List 1516.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2147,2025-01-31,13:10:00,2025-01-31 00:00:00,onyx_vnda,7.55,55.2,NaN,NaN,Stream Team Sample Log 2024_2025.xlsx,122.0,0.0,NaN,NaN,13.98,5.5,NaN
2148,2025-02-01,09:29:00,2025-02-01 00:00:00,adams_M1,8.35,300,NaN,NaN,Stream Team Sample Log 2024_2025.xlsx,123.0,0.4,NaN,NaN,13.54,-0.27,NaN
2149,2025-02-01,NaN,2025-02-01 00:00:00,miers_M2,7.96,100,NaN,NaN,Stream Team Sample Log 2024_2025.xlsx,124.0,1.1,NaN,NaN,13.14,-0.25,NaN
2150,2025-02-01,NaN,2025-02-01 00:00:00,garwood_colleen,NaN,NaN,NaN,NaN,Stream Team Sample Log 2024_2025.xlsx,125.0,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
def clean_time(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    val = val.replace(".0", "")
    val = val.replace(" ", "")
    if ":" in val:
        parts = val.split(":")
        if len(parts) == 2:
            hour = parts[0].zfill(2)
            minute = parts[1].zfill(2)
            return f"{hour}:{minute}"
    if val.isdigit():
        if len(val) <= 2:
            return f"{val.zfill(2)}:00"
        elif len(val) == 3:
            return f"0{val[0]}:{val[1:]}"
        elif len(val) == 4:
            return f"{val[:2]}:{val[2:]}"
    return np.nan

In [10]:
df["sample_time"] = df["sample_time"].apply(clean_time)
df

,Date Collected,sample_time,Date Filtered,sample_name,ph,conductivity,Comments,Unnamed: 7,source_file,Sample #,temperature,Unnamed: 9,Unnamed: 11,dissolved_oxygen,Turb (field),Unnamed: 13
0,2015-11-22,15:41,2015-11-23 00:00:00,Santa Fe Stream,8.38,NaN,Filtered one day late. Acidified two days late...,Several bottles found frozen in refrigerator -...,Stream Team Sample List 1516.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-11-22,19:57,2015-11-23 00:00:00,Wharton Creek,8.08,NaN,Filtered one day late. Acidified two days late,NaN,Stream Team Sample List 1516.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-11-24,15:33,2015-11-24 00:00:00,"C1, Commonwealth Stream",7.49,SC 96,Alkalinity bottle froze,NaN,Stream Team Sample List 1516.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-11-25,12:24,2015-11-25 00:00:00,"M1, Adams",NaN,NaN,NaN,NaN,Stream Team Sample List 1516.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-11-26,17:25,2015-11-26 00:00:00,"H1, Anderson Creek",6.13,AC 59.9,AC = Actual Conductivity - water too cold for ...,NaN,Stream Team Sample List 1516.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2147,2025-01-31,NaN,2025-01-31 00:00:00,onyx_vnda,7.55,55.2,NaN,NaN,Stream Team Sample Log 2024_2025.xlsx,122.0,0.0,NaN,NaN,13.98,5.5,NaN
2148,2025-02-01,NaN,2025-02-01 00:00:00,adams_M1,8.35,300,NaN,NaN,Stream Team Sample Log 2024_2025.xlsx,123.0,0.4,NaN,NaN,13.54,-0.27,NaN
2149,2025-02-01,NaN,2025-02-01 00:00:00,miers_M2,7.96,100,NaN,NaN,Stream Team Sample Log 2024_2025.xlsx,124.0,1.1,NaN,NaN,13.14,-0.25,NaN
2150,2025-02-01,NaN,2025-02-01 00:00:00,garwood_colleen,NaN,NaN,NaN,NaN,Stream Team Sample Log 2024_2025.xlsx,125.0,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
site_keywords = {
    'F1_Canada': ['canada'],
    'F2_Huey': ['huey'],
    'F3_LostSeal': ['lostseal', 'lost seal'],
    'F4_McKnight': ['mcknight'],
    'F5_Aiken': ['aiken'],
    'F6_VonGuerard': ['vonguerard', 'von guerard', 'f6'],
    'F7_Harnish': ['harnish'],
    'F8_Crescent': ['crescent'],
    'F9_Green': ['green'],
    'F10_Delta': ['delta'],
    'H1_Andersen': ['andersen', 'anderson'],
    'B3_Lawson': ['lawson'],
    'B5_Bohner': ['bohner'],
    'Onyx@Vanda': ['vanda'],
    'Onyx@LWRT': ['lwrt', 'lowerwright', 'wright', 'onyx_vnda'],
    'M1_Adams': ['adams'],
    'M2_MiersOutlet': ['outlet', 'm2'],
    'C1_Commonwealth': ['commonwealth', 'common'],
    'RelictChannel': ['relict', 'tributary', 'trib', 'f11'],
    'Mariah': ['mariah'],
    'Bowles': ['bowles'],
    'H2_House': ['house'],
    'Wharton': ['wharton'],
    'McKay': ['mckay'],
    'B1_Priscu': ['priscu'],
    'B2_SantaFe': ['santa', 'santafe'],
    'Sharp': ['sharp'],
    'B4_Lyons': ['lyons'],
    'BloodFalls': ['red', 'bfalls', 'blood'],
    'Lizotte': ['lizotte'],
    'Bartlette': ['bartlette'],
    'Vincent': ['vincent'],
    'Mason': ['mason'],
    'MiersInlet': ['inlet'],
    'Garwood@Colleen': ['colleen'],
    'GarwoodOutlet': ['blwglcr','below']
}

def standardize_site(name):
    if pd.isna(name):
        return np.nan
    name = str(name).lower().strip()
    for standard_name, keywords in site_keywords.items():
        for keyword in keywords:
            if keyword in name:
                return standard_name
    return name

df["sample_name"] = df["sample_name"].apply(standardize_site)
df

,Date Collected,sample_time,Date Filtered,sample_name,ph,conductivity,Comments,Unnamed: 7,source_file,Sample #,temperature,Unnamed: 9,Unnamed: 11,dissolved_oxygen,Turb (field),Unnamed: 13
0,2015-11-22,15:41,2015-11-23 00:00:00,B2_SantaFe,8.38,NaN,Filtered one day late. Acidified two days late...,Several bottles found frozen in refrigerator -...,Stream Team Sample List 1516.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-11-22,19:57,2015-11-23 00:00:00,Wharton,8.08,NaN,Filtered one day late. Acidified two days late,NaN,Stream Team Sample List 1516.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-11-24,15:33,2015-11-24 00:00:00,C1_Commonwealth,7.49,SC 96,Alkalinity bottle froze,NaN,Stream Team Sample List 1516.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-11-25,12:24,2015-11-25 00:00:00,M1_Adams,NaN,NaN,NaN,NaN,Stream Team Sample List 1516.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-11-26,17:25,2015-11-26 00:00:00,H1_Andersen,6.13,AC 59.9,AC = Actual Conductivity - water too cold for ...,NaN,Stream Team Sample List 1516.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2147,2025-01-31,NaN,2025-01-31 00:00:00,Onyx@LWRT,7.55,55.2,NaN,NaN,Stream Team Sample Log 2024_2025.xlsx,122.0,0.0,NaN,NaN,13.98,5.5,NaN
2148,2025-02-01,NaN,2025-02-01 00:00:00,M1_Adams,8.35,300,NaN,NaN,Stream Team Sample Log 2024_2025.xlsx,123.0,0.4,NaN,NaN,13.54,-0.27,NaN
2149,2025-02-01,NaN,2025-02-01 00:00:00,M2_MiersOutlet,7.96,100,NaN,NaN,Stream Team Sample Log 2024_2025.xlsx,124.0,1.1,NaN,NaN,13.14,-0.25,NaN
2150,2025-02-01,NaN,2025-02-01 00:00:00,Garwood@Colleen,NaN,NaN,NaN,NaN,Stream Team Sample Log 2024_2025.xlsx,125.0,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df = df.rename(columns={
    "Date Collected": "date_collected",
    "Date Filtered": "date_filtered",
    "Turb (field)": "turbidity"
})

final_columns = [
    "date_collected",
    "sample_time",
    "date_filtered",
    "sample_name",
    "ph",
    "conductivity",
    "temperature",
    "dissolved_oxygen",
    "turbidity",
    "source_file",
    "Comments"
]

final_df = df[final_columns].copy()

final_df = final_df.rename(columns={
    "Comments": "comments"
})

output_path = data_path / "master_sample_log.csv"

final_df.to_csv(output_path, index=False)

print(f"CSV exported to:\n{output_path}")

CSV exported to:
C:\Users\jared\Downloads\Sample Logs\master_sample_log.csv
